In [3]:
import random
from collections import deque

# Class representing a book in the library
class Book:
    def __init__(self, isbn: str, title: str, author: str):
        """
        Initialize a Book object with its ISBN, title, author, and availability status.
        A reservation queue is also maintained for members waiting for this book.
        """
        self.isbn = isbn
        self.title = title
        self.author = author
        self.available = True  # By default, books are available
        self.reservation_queue = deque()  # Queue for managing reservations

    def __str__(self):
        """
        String representation of the book for readability in logs or outputs.
        """
        return f"Title: {self.title}, Author: {self.author}, ISBN: {self.isbn}"

# Class representing a library member
class Member:
    def __init__(self, id: str, name: str, email: str, fines: float = 0.0):
        """
        Initialize a Member object with their ID, name, email, and outstanding fines.
        Each member also has a record of books they have borrowed.
        """
        self.id = id
        self.name = name
        self.email = email
        self.borrowed_books = []  # List of borrowed books
        self.fines = fines  # Outstanding fines

    def __str__(self):
        """
        String representation of the member, including borrowed books and fines.
        """
        books_str = "".join([f"\t{book.title} by {book.author}\n" for book in self.borrowed_books])
        return f"Member ID: {self.id}, Name: {self.name}, Email: {self.email}, Borrowed Books:\n{books_str}, Fines: {self.fines}"

    def get_borrowed_books(self):
        """
        Returns a formatted string of the books the member has borrowed.
        """
        books_str = "\n".join([f"{book.title} by {book.author}" for book in self.borrowed_books])
        return f"Books borrowed:\n{books_str}"

    def add_borrowed_book(self, book: Book):
        """
        Adds a book to the member's list of borrowed books.
        """
        self.borrowed_books.append(book)

    def remove_borrowed_book(self, book: Book):
        """
        Removes a book from the member's list of borrowed books.
        """
        self.borrowed_books.remove(book)

# Class to simulate an email server for notifications
class FakeSMTPServer:
    def __init__(self):
        """
        Initializes a fake SMTP server for logging emails sent during the simulation.
        Emails are stored in a list for later review or debugging.
        """
        self.sent_emails = []

    def sendmail(self, from_addr: str, to_addrs: str, msg: str):
        """
        Simulates sending an email by adding it to the list of sent emails.
        """
        self.sent_emails.append({'from': from_addr, 'to': to_addrs, 'msg': msg})

    def print_sent_emails(self):
        """
        Prints all emails that have been sent for review.
        """
        for email in self.sent_emails:
            print(f"From: {email['from']}\nTo: {email['to']}\nMessage: {email['msg']}\n")

    def quit(self):
        """
        Placeholder for closing the SMTP server, if needed.
        """
        pass

# Function to send emails to members
def send_notification_email(member: Member, message: str, fake_smtp_server: FakeSMTPServer):
    """
    Sends a notification email to a member using the fake SMTP server.
    Args:
        member (Member): The recipient of the email.
        message (str): The content of the email.
        fake_smtp_server (FakeSMTPServer): The server handling the email.
    """
    from_addr = "library@example.com"  # Simulated sender's address
    to_addrs = member.email  # Recipient's email address
    msg = f"Dear {member.name},\n\n{message}\n\nSincerely,\nThe Library Team"  # Email content
    fake_smtp_server.sendmail(from_addr, to_addrs, msg)

# Class representing the library system
class Library:
    def __init__(self, name: str):
        """
        Initializes the library with its name, a collection of books, and a list of members.
        """
        self.name = name
        self.books = {}  # Dictionary of books, keyed by ISBN
        self.members = {}  # Dictionary of members, keyed by ID

    def add_book(self, book: Book):
        """
        Adds a book to the library's collection.
        Args:
            book (Book): The book to be added.
        """
        self.books[book.isbn] = book

    def add_member(self, member: Member):
        """
        Adds a member to the library's membership list.
        Args:
            member (Member): The member to be added.
        """
        self.members[member.id] = member

    def reserve_book(self, member_id: str, isbn: str):
        """
        Allows a member to reserve a book if it's currently unavailable.
        Args:
            member_id (str): The ID of the member making the reservation.
            isbn (str): The ISBN of the book to be reserved.
        """
        if isbn in self.books and member_id in self.members:
            book = self.books[isbn]
            book.reservation_queue.append(member_id)

    def borrow_book(self, member_id: str, isbn: str, fake_smtp_server: FakeSMTPServer):
        """
        Handles the borrowing of a book by a member.
        Sends a notification if the transaction is successful.
        """
        if member_id not in self.members or isbn not in self.books:
            return False
        book = self.books[isbn]
        member = self.members[member_id]

        if not book.available:
            # Check if the member is first in the reservation queue
            if book.reservation_queue and book.reservation_queue[0] == member_id:
                book.reservation_queue.popleft()  # Remove from queue
                book.available = False
                member.add_borrowed_book(book)
                message = f"Your reserved book '{book.title}' is now available for you to borrow."
                send_notification_email(member, message, fake_smtp_server)
                return True
            return False

        # Borrow the book if it's available
        book.available = False
        member.add_borrowed_book(book)
        message = f"Thank you, {member.name} (Member #{member_id}), for borrowing '{book.title}'. Enjoy reading!"
        send_notification_email(member, message, fake_smtp_server)
        return True

    def return_book(self, member_id: str, isbn: str, fake_smtp_server: FakeSMTPServer):
        """
        Handles the return of a book by a member.
        Sends a notification about late fees or punctuality.
        """
        if member_id not in self.members or isbn not in self.books:
            return False
        member = self.members[member_id]
        book = self.books[isbn]
        if book not in member.borrowed_books:
            return False

        # Calculate late fee (randomized for simulation)
        days_late = random.randint(0, 5)
        late_fee = days_late * 0.50
        member.fines += late_fee
        member.remove_borrowed_book(book)
        book.available = True

        # Notify about the return status
        if late_fee > 0:
            message = (
                f"We understand things happen, {member.name}. You have been charged £{late_fee:.2f} "
                f"for a late return. Your total fines are now £{member.fines:.2f}. Please pay at least £{member.fines * 0.5:.2f}."
            )
        else:
            message = (
                f"Thank you for returning '{book.title}' on time, {member.name}. We appreciate your punctuality! "
                f"Your total fines remain at £{member.fines:.2f}. Keep up the great borrowing habits!"
            )
        send_notification_email(member, message, fake_smtp_server)

        # Notify next in line if the book is reserved
        if book.reservation_queue:
            next_member_id = book.reservation_queue.popleft()
            next_member = self.members[next_member_id]
            message = f"The book '{book.title}' is now available for you to borrow. Please collect it soon!"
            send_notification_email(next_member, message, fake_smtp_server)
        return True

# Helper functions for library simulation
def populate_library_books(library):
    """
    Populates the library with a predefined set of books.
    """
    titles = [
        "The Hunger Games", "To Kill a Mockingbird", "1984", "Old Man's War",
        "Animal Farm", "Lord of the Flies", "Brave New World", "Of Mice and Men", "The Grapes of Wrath"
    ]
    authors = [
        "Suzanne Collins", "Harper Lee", "George Orwell", "John Scalzi",
        "George Orwell", "William Golding", "Aldous Huxley", "John Steinbeck", "John Steinbeck"
    ]
    for i in range(len(titles)):
        book = Book(str(i + 1), titles[i], authors[i])
        library.add_book(book)

def populate_members(library):
    """
    Populates the library with a predefined set of members.
    """
    names = ["Alice Adams", "Benny Black", "Cathy Crawford", "Danny Davidson", "Eddie Evans",
             "Felicity Fox", "Gina Green", "Harry Hayes", "Isabelle Ingram", "Jackie Johnson"]
    for i, name in enumerate(names):
        email = f"{name.replace(' ', '')}@keel_library.co.uk"
        member = Member(str(i + 1), name, email)
        library.add_member(member)

def borrow_random_book(library, fake_smtp_server):
    """
    Simulates a random book borrowing by a random member.
    """
    member_ids = list(library.members.keys())
    book_isbns = list(library.books.keys())
    member_id = random.choice(member_ids)
    book_isbn = random.choice(book_isbns)
    library.borrow_book(member_id, book_isbn, fake_smtp_server)

def return_random_book(library, fake_smtp_server):
    """
    Simulates a random book return by a random member.
    """
    member_ids = list(library.members.keys())
    member_id = random.choice(member_ids)
    member = library.members[member_id]
    if member.borrowed_books:
        book = random.choice(member.borrowed_books)
        library.return_book(member_id, book.isbn, fake_smtp_server)

# Main simulation function
def main():
    """
    Runs the library simulation, including borrowing and returning books.
    """
    library = Library("Rebelky Elite")  # Initialize library
    fake_smtp_server = FakeSMTPServer()  # Initialize fake email server
    populate_library_books(library)  # Add books to the library
    populate_members(library)  # Add members to the library

    # Simulate borrowing books
    for _ in range(10):
        borrow_random_book(library, fake_smtp_server)

    # Simulate returning books
    for _ in range(10):
        return_random_book(library, fake_smtp_server)

    # Print all sent emails
    fake_smtp_server.print_sent_emails()

# Run the simulation
main()


From: library@example.com
To: BennyBlack@keel_library.co.uk
Message: Dear Benny Black,

Thank you, Benny Black (Member #2), for borrowing 'The Hunger Games'. Enjoy reading!

Sincerely,
The Library Team

From: library@example.com
To: BennyBlack@keel_library.co.uk
Message: Dear Benny Black,

Thank you, Benny Black (Member #2), for borrowing '1984'. Enjoy reading!

Sincerely,
The Library Team

From: library@example.com
To: JackieJohnson@keel_library.co.uk
Message: Dear Jackie Johnson,

Thank you, Jackie Johnson (Member #10), for borrowing 'Old Man's War'. Enjoy reading!

Sincerely,
The Library Team

From: library@example.com
To: HarryHayes@keel_library.co.uk
Message: Dear Harry Hayes,

Thank you, Harry Hayes (Member #8), for borrowing 'Lord of the Flies'. Enjoy reading!

Sincerely,
The Library Team

From: library@example.com
To: AliceAdams@keel_library.co.uk
Message: Dear Alice Adams,

Thank you, Alice Adams (Member #1), for borrowing 'To Kill a Mockingbird'. Enjoy reading!

Sincerely,
Th